In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data = pd.read_csv('/content/drive/MyDrive/GSE115469_Data.csv', index_col=0)

In [21]:
data.shape

(20007, 8444)

In [15]:
data.isna().values.any()

np.False_

# PRETPROCESIRANJE - Opsti koraci

Predlog za pretprocesiranje:

OPSTE
  - uklanjanje neinformativnih gena
    - geni koji se eksprimuju u manje od 5% celija
    - geni koji imaju skoro isti nivo ekspresije u svakoj celiji
  - clipping autlajera
  - transponovanje matrice

PRAVILA PRIDRUZIVANJA
  - promena formata u transakcije -> Cell1 = {Gene1, Gene2, Gene5}
    - gen je prisutan u celiji ako ekspresija prelazi odredjeni threshold (npr. 75. percentil gena)
  - binarizacija
  - uklanjanje previse cestih gena (cela support/lift/coverage analiza)
  - uklanjanje previse retkih gena

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;APRIORI
- feature selection - izdvojiti najvarijabilnije gene, npr. 2000 njih

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;ECLAT
- vertical data layout

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;CHARM
- support threshold? (valjda je vec obradjeno)

## Korak 1: Uklanjanje neinformativnih gena

### 1.1 Geni koji se eksprimuju u manje od 5% celija

In [23]:
gene_frequency = (data > 0).mean(axis=1)
data = data.loc[gene_frequency >= 0.05]

In [27]:
data.shape

(6717, 8444)

In [24]:
# num_of_cells = data.shape[1]
# five_percent = np.floor((num_of_cells / 100) * 5)

In [25]:
# def no_of_cells_exp(data, gene):
#   col = data.loc[gene]
#   return np.sum(col.astype(bool))

In [26]:
# for gene in data.index:
#   if no_of_cells_exp(data, gene) <= five_percent:
#     data.drop(gene, axis=0)

### 1.2 Geni koji imaju skoro isti nivo ekspresije u svakoj celiji

In [40]:
gene_var = data.var(axis=1)
gene_var_sorted = gene_var.sort_values(ascending=False)
gene_var_keep = gene_var_sorted[:2000]

In [41]:
gene_var_keep

,0
APOC3,12.340822
APOA2,11.126836
ORM1,9.849425
HP,9.393121
ALB,9.389534
...,...
TACC1,0.375015
PDXDC1,0.374714
FUNDC2,0.374669
RNF167,0.374545


In [42]:
data = data.loc[gene_var_keep.index]

In [43]:
data.shape

(2000, 8444)

## Korak 2: Outlier clipping

In [44]:
data = data.clip(upper=data.quantile(0.999).max())

## Korak 3: Transponovanje matrice

In [45]:
data = data.T

In [46]:
data.shape

(8444, 2000)

# PRETPROCESIRANJE - Pravila pridruzivanja

In [52]:
data_AR = data